# 3MDN Model: Structured Mixture Density Network

本版采用以下数据流程：

1. 直接读取 `1DataDownload_Clean.ipynb` 生成的两张表，不重复生成 target、momentum 或 rolling features。
2. Micro 使用显式特征清单；`target_ret_final` 只作为标签，并通过断言阻止泄漏。
3. Macro 在初始训练期拟合尺度，之后只使用每个时间块开始前的历史月份更新尺度。
4. 自动识别 local、Colab 和 Kaggle；`quick` 模式仅抽取少量股票和前两个 OOS 窗口。
5. Warm start 保留历史参数状态；若使用 rolling window，应将其解释为 online fine-tuning，而不是完全遗忘旧样本的重新估计。

## Planned Robustness Checks

1. Normal versus Student-$t$ components.
2. $K=1$ versus multiple mixture components.
3. Rank versus normal-score micro normalization.
4. With and without the macro/FiLM channel.
5. Rolling versus expanding windows and uniform versus time-decay weights.

In [ ]:
from pathlib import Path
import importlib.util
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

ENV_OVERRIDE = None  # None, 'local', 'colab', or 'kaggle'
KAGGLE_DATASET_SLUG = 'mdndata-1990'
COLAB_DATA_DIR = Path('/content/drive/MyDrive/PricingTailRisk_MDN/data')
COLAB_OUTPUT_DIR = Path('/content/drive/MyDrive/PricingTailRisk_MDN/output')

run_test = 'quick'  # 'quick' 或 'full'
random_seed = 42

best_k = 3  # set to 1 for the nested K=1 specification

initial_train_months = 180
val_months = 12
test_months = 12

# 初始窗口完整训练；之后每个窗口在原模型参数上继续微调。
early_stopping_max_epochs = 100
first_window_epochs = 100
later_window_epochs = 50
batch_size = 1024
lr = 1e-3
patience = 20
dropout_rate = 0.10
sigma_floor = 0.005  # 0.5% monthly component-scale floor
quick_max_stocks = 500

micro_normalize = 'rank'  # 'normal' or 'rank' for accounting-ratio features
RAW_SCALE_MICRO_FEATURES = [
    'mthret', 'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log',
]
MICRO_FEATURES = [
    'mthret', 'bm', 'evm', 'pe_exi', 'ps', 'pcf', 'pfcf', 'npm',
    'opmad', 'gpm', 'cfm', 'roe', 'aftret_invcapx', 'gprof',
    'debt_at', 'debt_ebitda', 'short_debt', 'ocf_lct', 'de_ratio',
    'cash_ratio', 'curr_ratio', 'at_turn', 'rect_turn', 'pay_turn',
    'sale_equity', 'rd_sale', 'adv_sale', 'staff_sale', 'accrual',
    'MOM12m', 'mu_bench', 'sigma_bench', 'mthcap_log',
]
RANK_MICRO_FEATURES = [
    col for col in MICRO_FEATURES if col not in RAW_SCALE_MICRO_FEATURES
]
MACRO_FEATURES = [
    'PAYEMS', 'UNRATE', 'CLAIMSx', 'INDPRO', 'DPCERA3M086SBEA',
    'AMDMNOx', 'ISRATIOx', 'HOUST', 'M2SL', 'NONBORRES', 'BUSLOANS',
    'FEDFUNDS', 'CP3Mx', 'UEMPMEAN', 'CPIAUCSL', 'VIXCLSx', 'BAAFFM',
    'TERM_SPREAD', 'CREDIT_SPREAD',
]


macro_norm_update_months = test_months
normalization_eps = 1e-8

GLOBAL_DIST_TYPE = 't'       # 't' 或 'normal'
epochs_type = 'early_stopping'        # 'fixed' 或 'early_stopping'
windows_type = 'rolling'     # 'expanding' 或 'rolling'
weights_type = 'month_equal'  # 'uniform', 'month_equal', or 'time_decay'
SAVE_PROCESSED_PANEL = False
RELEASE_COLAB_RUNTIME = False

In [ ]:
def detect_environment():
    if ENV_OVERRIDE is not None:
        return ENV_OVERRIDE.lower()
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/input').exists():
        return 'kaggle'
    if importlib.util.find_spec('google.colab') is not None:
        return 'colab'
    return 'local'

def find_local_project_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'data' / 'micro_data.parquet').exists():
            return candidate.resolve()
    raise FileNotFoundError('Cannot locate the project root containing data/micro_data.parquet.')

def find_kaggle_input(filename):
    preferred = Path('/kaggle/input') / KAGGLE_DATASET_SLUG / filename
    if preferred.exists():
        return preferred
    matches = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f'{filename} was not found under /kaggle/input.')
    raise RuntimeError(f'Multiple Kaggle inputs named {filename}: {matches}')

env = detect_environment()
if env == 'local':
    PROJECT_ROOT = find_local_project_root()
    micro_path = PROJECT_ROOT / 'data' / 'micro_data.parquet'
    macro_path = PROJECT_ROOT / 'data' / 'MacroFactor_McCracken.parquet'
    output_dir = PROJECT_ROOT / 'output'
elif env == 'kaggle':
    PROJECT_ROOT = Path('/kaggle/working')
    micro_path = find_kaggle_input('micro_data.parquet')
    macro_path = find_kaggle_input('MacroFactor_McCracken.parquet')
    output_dir = Path('/kaggle/working')
elif env == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content')
    micro_path = COLAB_DATA_DIR / 'micro_data.parquet'
    macro_path = COLAB_DATA_DIR / 'MacroFactor_McCracken.parquet'
    output_dir = COLAB_OUTPUT_DIR
else:
    raise ValueError(f'Unsupported environment: {env}')

for data_path in (micro_path, macro_path):
    if not data_path.exists():
        raise FileNotFoundError(
            f'Missing {data_path}. On Colab put both parquet files in {COLAB_DATA_DIR}; '
            'on Kaggle attach the dataset containing both files.'
        )

micro_required = ['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + MICRO_FEATURES
macro_required = ['mthcaldt'] + MACRO_FEATURES
micro_df = pd.read_parquet(micro_path, columns=micro_required)
macro_df = pd.read_parquet(macro_path, columns=macro_required)
output_dir.mkdir(parents=True, exist_ok=True)
print({'environment': env, 'micro_path': str(micro_path), 'macro_path': str(macro_path), 'output_dir': str(output_dir)})

In [ ]:
print('Micro DataFrame shape:', micro_df.shape)
print('Micro months:', micro_df['mthcaldt'].nunique())
display(pd.concat([micro_df.head(5), micro_df.tail(5)]))
print("Macro DataFrame shape:", macro_df.shape)
display(pd.concat([macro_df.head(5), macro_df.tail(5)]))


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
import json
import copy
import warnings
import random
import time
from datetime import datetime
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

# warnings.filterwarnings('ignore')

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# =========================================================
# 1. Validate the cleaned micro table and normalize explicit features
# =========================================================
micro_cols = MICRO_FEATURES.copy()
target_col = 'target_ret_final'
required_micro = ['permno', 'mthcaldt', 'ffi49', target_col] + micro_cols
missing_micro = [col for col in required_micro if col not in micro_df.columns]
if missing_micro:
    raise KeyError(f'Missing micro columns: {missing_micro}')
if target_col in micro_cols:
    raise AssertionError('target_ret_final must be a label only, never a feature.')
if len(micro_cols) != len(set(micro_cols)):
    raise AssertionError('Duplicate micro feature names found.')

micro_df = micro_df[required_micro].copy()
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
micro_df['ffi49'] = micro_df['ffi49'].fillna(0).astype(int)
if micro_df.duplicated(['permno', 'mthcaldt']).any():
    raise ValueError('Duplicate (permno, mthcaldt) rows found in micro_data.')
if not micro_df['ffi49'].between(0, 49).all():
    raise ValueError('ffi49 must be an integer in [0, 49].')

print(f'>>> Applying {micro_normalize} normalization to {len(RANK_MICRO_FEATURES)} micro features...')
if micro_normalize == 'rank':
    micro_norm = (
        micro_df.groupby('mthcaldt')[RANK_MICRO_FEATURES].rank(pct=True) * 2.0 - 1.0
    )
elif micro_normalize == 'normal':
    monthly_mean = micro_df.groupby('mthcaldt')[RANK_MICRO_FEATURES].transform('mean')
    monthly_std = micro_df.groupby('mthcaldt')[RANK_MICRO_FEATURES].transform('std')
    micro_norm = (
        micro_df[RANK_MICRO_FEATURES] - monthly_mean
    ) / (monthly_std + normalization_eps)
else:
    raise ValueError("micro_normalize must be 'normal' or 'rank'")
micro_df[RANK_MICRO_FEATURES] = micro_norm.fillna(0.0).to_numpy()

feature_block = micro_df[micro_cols + [target_col]].replace([np.inf, -np.inf], np.nan)
if feature_block.isna().any().any():
    bad = feature_block.isna().sum()
    raise ValueError(f'Non-finite model values remain: {bad[bad > 0].to_dict()}')

micro_model_df = micro_df[
    ['permno', 'mthcaldt', 'ffi49', target_col] + micro_cols
].sort_values(['mthcaldt', 'permno']).reset_index(drop=True)

# Keep quick runs small enough for a local smoke test; ranks were computed on the full cross-section.
if run_test == 'quick':
    rng = np.random.default_rng(random_seed)
    all_ids = np.sort(micro_model_df['permno'].unique())
    selected_ids = rng.choice(
        all_ids, size=min(quick_max_stocks, len(all_ids)), replace=False
    )
    micro_model_df = micro_model_df.loc[
        micro_model_df['permno'].isin(selected_ids)
    ].reset_index(drop=True)

print('Micro model panel:', micro_model_df.shape)
display(pd.concat([micro_model_df.head(3), micro_model_df.tail(3)]))

In [ ]:
# =========================================================
# 2. Macro：初始训练期统一拟合，之后按时间块更新
# =========================================================
def standardize_macro(
    macro_data,
    macro_cols,
    model_dates,
    initial_months,
    update_months,
):
    macro = macro_data.set_index('mthcaldt').loc[model_dates, macro_cols].copy()
    macro_z = pd.DataFrame(index=model_dates, columns=macro_cols, dtype=float)

    initial_dates = model_dates[:initial_months]
    initial_mean = macro.loc[initial_dates].mean()
    initial_std = macro.loc[initial_dates].std(ddof=0)
    macro_z.loc[initial_dates] = (
        (macro.loc[initial_dates] - initial_mean) /
        (initial_std + normalization_eps)
    ).to_numpy()

    for start in range(initial_months, len(model_dates), update_months):
        # print(f'>>> Standardizing Macro features for block starting at {model_dates[start]}...')
        block_dates = model_dates[start:start + update_months]
        history = macro.iloc[:start]
        history_mean = history.mean()
        history_std = history.std(ddof=0)
        macro_z.loc[block_dates] = (
            (macro.loc[block_dates] - history_mean) /
            (history_std + normalization_eps)
        ).to_numpy()

    macro_z.index.name = 'mthcaldt'
    return macro_z.reset_index()

macro_cols = MACRO_FEATURES.copy()
required_macro = ['mthcaldt'] + macro_cols
missing_macro = [col for col in required_macro if col not in macro_df.columns]
if missing_macro:
    raise KeyError(f'Missing macro columns: {missing_macro}')
if 'target_ret_final' in macro_cols:
    raise AssertionError('The target cannot appear in macro features.')
macro_df = macro_df[required_macro].copy()
macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df = macro_df.sort_values('mthcaldt').reset_index(drop=True)
if macro_df['mthcaldt'].duplicated().any():
    raise ValueError('Duplicate months found in MacroFactor_McCracken.')

model_dates = pd.DatetimeIndex(np.sort(micro_model_df['mthcaldt'].unique()))

macro_model_df = standardize_macro(
    macro_data=macro_df,
    macro_cols=macro_cols,
    model_dates=model_dates,
    initial_months=initial_train_months,
    update_months=macro_norm_update_months,
)
pd.concat([macro_model_df.head(5), macro_model_df.tail(5)])

In [ ]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
processed_df = (
    micro_model_df
    .merge(macro_model_df, on='mthcaldt', how='left', validate='many_to_one')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

model_value_cols = micro_cols + macro_cols + ['target_ret_final']
if processed_df.duplicated(['permno', 'mthcaldt']).any():
    raise ValueError('Duplicate model-panel keys after the macro merge.')
if not np.isfinite(processed_df[model_value_cols].to_numpy(dtype=float)).all():
    raise ValueError('The merged model panel contains non-finite values.')
if 'target_ret_final' in micro_cols or 'target_ret_final' in macro_cols:
    raise AssertionError('Target leakage detected in the feature lists.')

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

if SAVE_PROCESSED_PANEL:
    processed_df.to_parquet(output_dir / f'processed_df_{run_test}.parquet', index=False)

pd.concat([processed_df.head(5), processed_df.tail(5)])

In [ ]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months + test_months <= len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

## Structured MDN with FiLM

The executable definition below is the production architecture. Obsolete experimental model definitions were removed to keep top-to-bottom execution unambiguous.


In [ ]:
# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + FiLM 调制
# ==========================================
class StructuredMDN_with_FiLM(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, 
                 micro_hidden_dim=64, macro_hidden_dim=32, num_components=3, 
                 dropout_rate=0.1, dist_type='normal'):
        """
        结构化 MDN (加入 FiLM 调制机制)
        - 保持 2D 截面输入，不引入序列记忆 (无 GRU)
        - 宏观特征不仅预测 Regime 概率，还直接调控微观特征的表达
        """
        super(StructuredMDN_with_FiLM, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        # ++++++++++++++++++++++++++++++++++++++++++
        # 新增：行业实体嵌入层 (Entity Embedding)
        # ++++++++++++++++++++++++++++++++++++++++++
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        
        # 计算拼接后的微观特征总维度 = 连续特征维度 + 行业嵌入维度(6)
        total_micro_dim = micro_dim + ind_emb_dim
        # ==========================================
        # 通道 A: Macro Network (标准 MLP)
        # 输入: (Batch, Macro_Dim) 截面宏观数据
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim , macro_hidden_dim//2), # 🌟 优化：增加一层，让宏观特征更丰富
            nn.LayerNorm(macro_hidden_dim//2),
            nn.ELU()
        )
        
        # 门控预测头：基于宏观隐状态预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim//2, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 目标是对齐微观网络的最后一层输出维度
        target_dim = 8
        
        # Gamma 负责“缩放(Scale)”微观特征，Beta 负责“平移(Shift)”微观特征
        self.film_gamma = nn.Linear(macro_hidden_dim//2, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim//2, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入: (Batch, Micro_Dim) 截面微观数据
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 防止截面前视偏差
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU()
        )
        
        # 预测头 (Heads): 注意，这里输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)
        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)    

        # ==========================================
        # 🌟 优化：MDN 安全初始化策略
        # ==========================================
        # 1. 均值头：初始化为极小的值，防止初始预期收益率过高
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        
        # 2. Sigma head starts near 9% monthly scale before learning.
        nn.init.constant_(self.sigma_head.bias, -2.35)
        
        if self.dist_type == 't':
            # 自由度头：初始化 nu 较大（例如 10），使其初始状态接近正态分布
            # 因为你的代码里有 + 2.01 的截断，softplus(x) + 2.01 = 10 -> x ≈ 2.07
            # nn.init.constant_(self.nu_head.bias, 2.07)  
            nn.init.constant_(self.nu_head.bias, 5)

    def forward(self, x_macro, x_micro, x_micro_ind):
        """
        x_macro: (Batch, Macro_Dim) - 当前时刻的宏观因子 (2D张量)
        x_micro: (Batch, Micro_Dim) - 当前时刻的微观因子 (2D张量)
        x_micro_ind: (Batch,) - 行业整数索引 0~47 (Long Tensor)
        """
        # ------------------------------------------
        # 1. 宏观特征提取
        # ------------------------------------------
        h_macro = self.macro_net(x_macro) # 形状: (Batch, macro_hidden_dim)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数 (根据当前宏观环境)
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ++++++++++++++++++++++++++++++++++++++++++
        # 3. 新增：Embedding 降维与特征拼接
        # ++++++++++++++++++++++++++++++++++++++++++
        # 将 [0, 1, ..., 49] 的整数索引，转化为 6 维稠密向量
        ind_emb = self.ind_embedding(x_micro_ind) # 形状: (Batch, 6)
        
        # 在特征维度 (dim=-1) 上拼接连续特征和行业 Embedding
        x_micro_combined = torch.cat([x_micro, ind_emb], dim=-1)
        
        # 修改：将拼接后的组合特征送入提取器
        h_micro = self.micro_extractor(x_micro_combined)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观环境直接介入微观特征的表达。
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        # Mu 路径：直接使用纯粹的、未受宏观污染的微观特征
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + sigma_floor
        
        # 3. 🌟 根据开关决定 Nu
        nu = None # 默认为空
        if self.dist_type == 't':
            # 保证自由度严格大于 2，以确保方差存在
            nu = nn.functional.softplus(self.nu_head(h_modulated)) + 2.01
            
        # 统一返回4个值，如果是 normal，nu 就是 None
        return pi_logits, mu, sigma, nu


## Weighted likelihood and training

Training uses explicit month-aware sample weights and device-side batch indexing for Kaggle/Colab GPU runs.


In [ ]:

# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type='normal'):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

# 🌟 核心开关：底层分布切换
    if dist_type == 't':
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    else:
        dist = torch.distributions.Normal(mu, sigma)
    
    # 分布的 log_prob
    log_dist = dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_dist
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.sum()/ weights.sum()

In [ ]:
# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def tensor_batch_indices(length, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(length, device=device)
    else:
        indices = torch.arange(length, device=device)
    for start in range(0, length, batch_size):
        yield indices[start:start + batch_size]

def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=100, batch_size=1024, lr=1e-3, patience=25): 
    if run_test == "quick":
        epochs = 2
        batch_size = 512
        patience = 1

    # 2. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    has_validation = (y_val is not None)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    train_size = y_train.numel()
    train_loss_history = []

    # 3. 🌟 智能判定：只有当验证标签不为空时，才实例化验证集 Loader
    if has_validation:
        validation_size = y_val.numel()
        epochs_no_improve = 0
        best_model_state = None
    best_val_loss = float('inf')
    val_loss_history = []
    
    
    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            train_size, batch_size, X_train_macro.device, shuffle=True
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx].unsqueeze(1)
            b_w = w_train[batch_idx].unsqueeze(1)
            optimizer.zero_grad()
            
            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            optimizer.step()
            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight
            
        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)
        
        # Validation
        if has_validation:
            model.eval()
            val_loss_numerator = 0.0
            val_weight_sum = 0.0
            with torch.no_grad():
                for batch_idx in tensor_batch_indices(
                    validation_size, batch_size, X_val_macro.device, shuffle=False
                ):
                    b_macro = X_val_macro[batch_idx]
                    b_micro = X_val_micro[batch_idx]
                    b_industry = X_val_industry[batch_idx]
                    b_y = y_val[batch_idx].unsqueeze(1)
                    b_w = w_val[batch_idx].unsqueeze(1)
                    pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
                    loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)
                    batch_weight = float(b_w.sum().item())
                    val_loss_numerator += loss.item() * batch_weight
                    val_weight_sum += batch_weight
            val_loss = val_loss_numerator / max(val_weight_sum, 1e-12)
            val_loss_history.append(val_loss)

            scheduler.step(val_loss)
                
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
    if has_validation and best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

In [ ]:
# ==========================================
# 6. 固定 epoch 的 warm-start 训练：第一期 100 个 epoch，后续窗口 50 个 epoch
# ==========================================
def train_structured_mdn_fixed_epochs(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         first_window_epochs=100, later_window_epochs=50,
                         batch_size=1024, lr=1e-3, patience=25, is_first_window=False):

    # ==========================================
    # 每个窗口都在传入模型的现有参数上继续训练。
    # Train + Val 合并训练；第一轮 100 epoch，后续窗口 50 epoch。
    # ==========================================
    if run_test == 'quick':
        epochs_to_run = 2 if is_first_window else 1
    else:
        epochs_to_run = first_window_epochs if is_first_window else later_window_epochs

    combined_macro = torch.cat([X_train_macro, X_val_macro], dim=0)
    combined_micro = torch.cat([X_train_micro, X_val_micro], dim=0)
    combined_industry = torch.cat([X_train_industry, X_val_industry], dim=0)
    combined_y = torch.cat([y_train, y_val], dim=0)
    combined_w = torch.cat([w_train, w_val], dim=0)
    combined_size = combined_y.numel()

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.8, patience=8, min_lr=1e-6,
    )
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs_to_run):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            combined_size, batch_size, combined_macro.device, shuffle=True
        ):
            b_macro = combined_macro[batch_idx]
            b_micro = combined_micro[batch_idx]
            b_industry = combined_industry[batch_idx]
            b_y = combined_y[batch_idx].unsqueeze(1)
            b_w = combined_w[batch_idx].unsqueeze(1)
            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight
            
        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)
        val_loss_history.append(train_loss)   # 无独立 val，用 train_loss 占位
        scheduler.step(train_loss)

    return model, train_loss, train_loss_history, val_loss_history

In [ ]:
# ==========================================
# Sample weights: prevent months with more listed stocks from dominating
# ==========================================
def calculate_sample_weights(df_dates, mode='month_equal', half_life_months=36):
    dates = pd.to_datetime(df_dates).reset_index(drop=True)
    if mode == 'uniform':
        weights = np.ones(len(dates), dtype=np.float64)
    elif mode in {'month_equal', 'time_decay'}:
        month_counts = dates.groupby(dates).transform('size').to_numpy(dtype=float)
        weights = 1.0 / np.maximum(month_counts, 1.0)
        if mode == 'time_decay':
            max_date = dates.max()
            months_diff = (
                (max_date.year - dates.dt.year) * 12
                + (max_date.month - dates.dt.month)
            ).to_numpy(dtype=float)
            weights *= np.power(0.5, months_diff / half_life_months)
    else:
        raise ValueError("weights_type must be 'uniform', 'month_equal', or 'time_decay'")
    weights = weights / weights.mean()
    return weights.astype(np.float32)

In [ ]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)
    print()
    print(">>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")

    # 1. 实例化时间窗口生成器。索引必须与全局 Tensor 一一对应。
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    if windows_type == 'expanding':
        window_gen = generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )
    else:
        window_gen = generator.rolling_window_generator(
            train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    # Micro 和 Macro 已在窗口循环前完成标准化。
    # "Pushing the full dataset into device memory once..."
    global_X_mac = torch.from_numpy(
        processed_df[macro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_X_mic = torch.from_numpy(
        processed_df[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)
    global_ind = torch.from_numpy(
        processed_df['ffi49'].to_numpy(dtype=np.int64, copy=True)
    ).to(device)
    global_Y = torch.from_numpy(
        processed_df['target_ret_final'].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    history_logs = []
    all_oos_predictions = []
    macro_dim = len(macro_cols)
    micro_dim = len(micro_cols)
    
    # 模型只初始化一次。rolling 窗口之间保留参数，下一期继续微调。
    global_model = StructuredMDN_with_FiLM(
        macro_dim,
        micro_dim,
        num_industries=50,
        ind_emb_dim=6,
        num_components=best_k,
        dist_type=GLOBAL_DIST_TYPE,
        dropout_rate=dropout_rate,
    ).to(device)

    print(
        f">>> [3/3] Starting {windows_type.title()} Window Structured MDN "
        "Training with warm start..."
    )

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx + 1}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )
        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = (
            global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        )
        X_va_mic, X_va_ind, Y_va = (
            global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        )
        X_te_mic, X_te_ind = global_X_mic[idx_te], global_ind[idx_te]

        # Month-equal weighting prevents months with more firms from dominating.
        if epochs_type == 'fixed':
            combined_dates = pd.concat(
                [train_df['mthcaldt'], val_df['mthcaldt']], ignore_index=True
            )
            combined_weights = calculate_sample_weights(
                combined_dates, mode=weights_type, half_life_months=36
            )
            w_train_np = combined_weights[:len(train_df)]
            w_val_np = combined_weights[len(train_df):]
        else:
            w_train_np = calculate_sample_weights(
                train_df['mthcaldt'], mode=weights_type, half_life_months=36
            )
            w_val_np = calculate_sample_weights(
                val_df['mthcaldt'], mode=weights_type, half_life_months=36
            )

        w_train = torch.tensor(w_train_np, dtype=torch.float32, device=device)
        w_val = torch.tensor(w_val_np, dtype=torch.float32, device=device)

        current_lr = lr if window_idx == 0 else lr * 0.3

        is_first_window = (window_idx == 0)

        if epochs_type == 'fixed':
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn_fixed_epochs(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                first_window_epochs=first_window_epochs,
                later_window_epochs=later_window_epochs,
                patience=patience,
                batch_size=batch_size,
                is_first_window=is_first_window,
            )
        else:
            global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac,
                X_train_micro=X_tr_mic,
                X_train_industry=X_tr_ind,
                y_train=Y_tr,
                w_train=w_train,
                X_val_macro=X_va_mac,
                X_val_micro=X_va_mic,
                X_val_industry=X_va_ind,
                y_val=Y_va,
                w_val=w_val,
                lr=current_lr,
                epochs=early_stopping_max_epochs,
                patience=patience,
                batch_size=batch_size,
            )

            # 基于 Train + Val 联合集合做短暂 fine-tune。
            epochs_run = len(tr_hist)
            finetune_epochs = 2 if run_test == 'quick' else max(5, epochs_run // 10)
            print(
                f"  Phase 1 done: epochs_run={epochs_run}, "
                f"finetune_epochs={finetune_epochs}"
            )

            X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
            X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
            X_tr_ind_ft = torch.cat([X_tr_ind, X_va_ind], dim=0)
            Y_tr_ft = torch.cat([Y_tr, Y_va], dim=0)

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            w_ft_np = calculate_sample_weights(
                train_val_df['mthcaldt'], mode=weights_type, half_life_months=36
            )
            w_ft = torch.tensor(w_ft_np, dtype=torch.float32, device=device)

            global_model, _, _, _ = train_structured_mdn(
                model=global_model,
                X_train_macro=X_tr_mac_ft,
                X_train_micro=X_tr_mic_ft,
                X_train_industry=X_tr_ind_ft,
                y_train=Y_tr_ft,
                w_train=w_ft,
                lr=current_lr * 0.5,
                epochs=finetune_epochs,
                patience=finetune_epochs + 1,
            )
            print(
                f"  Phase 2 fine-tune done "
                f"({finetune_epochs} epochs on Train+Val)"
            )

        duration = time.time() - start_time
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist,
            'val_loss_trace': va_hist,
        })

        # 样本外预测。
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(
                X_te_mac, X_te_mic, X_te_ind
            )
            pi = torch.softmax(pi_logits, dim=-1)

        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['model_id'] = f'mdn_{GLOBAL_DIST_TYPE}_k{best_k}'
        preds_df['pi_vec'] = [
            json.dumps(vec.tolist()) for vec in pi.cpu().numpy()
        ]
        preds_df['pi_logits_vec'] = [
            json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()
        ]
        preds_df['mu_vec'] = [
            json.dumps(vec.tolist()) for vec in mu.cpu().numpy()
        ]
        preds_df['sigma_vec'] = [
            json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()
        ]

        if nu is not None:
            preds_df['nu_vec'] = [
                json.dumps(vec.tolist()) for vec in nu.cpu().numpy()
            ]
        else:
            preds_df['nu_vec'] = None

        all_oos_predictions.append(preds_df)

        if run_test == "quick" and window_idx == 1:
            break

    # 提取并保存行业 Embedding。
    embedding_weights = global_model.ind_embedding.weight.detach().cpu().numpy()
    df_emb = pd.DataFrame(
        embedding_weights,
        columns=[f'emb_dim_{i + 1}' for i in range(6)],
    )
    df_emb['ffi49'] = df_emb.index

    print()
    print(">>> Pipeline complete! Aggregating all Out-of-Sample predictions...")

    if not all_oos_predictions:
        raise RuntimeError('No OOS windows were produced. Check the window lengths and dates.')
    final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
    if final_oos_df.duplicated(['model_id', 'permno', 'mthcaldt']).any():
        raise ValueError('Duplicate model-stock-month predictions found.')
    metrics_df = pd.DataFrame(history_logs)
    print(f"Successfully generated {len(final_oos_df)} predictions.")

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
run_tag = f'mdn_{GLOBAL_DIST_TYPE}_k{best_k}_{windows_type}_{run_test}_{timestamp}'
forecast_path = output_dir / f'final_oos_{run_tag}.parquet'
metrics_path = output_dir / f'metrics_{run_tag}.parquet'
embedding_path = output_dir / f'industry_embeddings_{run_tag}.parquet'
config_path = output_dir / f'config_{run_tag}.json'

final_oos_df.to_parquet(forecast_path, index=False)
metrics_df.to_parquet(metrics_path, index=False)
df_emb.to_parquet(embedding_path, index=False)
run_config = {
    'environment': env,
    'micro_path': str(micro_path),
    'macro_path': str(macro_path),
    'run_test': run_test,
    'seed': random_seed,
    'num_components': best_k,
    'distribution': GLOBAL_DIST_TYPE,
    'windows_type': windows_type,
    'weights_type': weights_type,
    'micro_normalize': micro_normalize,
    'micro_features': micro_cols,
    'macro_features': macro_cols,
    'initial_train_months': initial_train_months,
    'validation_months': val_months,
    'test_months': test_months,
    'sigma_floor': sigma_floor,
}
config_path.write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print('Saved forecasts:', forecast_path)
print('Saved metrics:', metrics_path)
print('Saved embeddings:', embedding_path)
print('Saved config:', config_path)

In [ ]:
if env == 'colab' and RELEASE_COLAB_RUNTIME:
    from google.colab import runtime
    runtime.unassign()